# SGConv Graph Neural Network for Fake News Detection
## UPFD Dataset with Combined Features

This notebook trains an SGConv model on the UPFD (User-Preference-aware Fake-news Detection) dataset with combined features (spacy, bert, content, profile).

## 1. Install Dependencies

In [1]:
!pip install torch torch-geometric scikit-learn pandas numpy scipy -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 58.9 MB/s eta 0:00:00


## 2. Import Libraries

In [2]:
import os.path as osp
import numpy as np
import pandas as pd
import torch
from torch_geometric.data import Data
from scipy.sparse import csr_matrix
from torch_geometric.nn import SGConv
import torch.nn as nn
import torch.optim as optim
from torch.nn import functional as F
from torch_geometric.nn import global_mean_pool
from torch_geometric.loader import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings('ignore')

## 3. Download and Load Data

For Kaggle, adjust the path to your dataset location

In [3]:
dataset_name = 'gossipcop'
feature_types = ['spacy', 'bert', 'content', 'profile']

data_dir = f'/kaggle/input/datasets/siddharth194/{dataset_name}'

print(f"Loading {dataset_name} dataset with combined features: {', '.join(feature_types)}")
print("-" * 60)

# Load core data
graph_labels = np.load(osp.join(data_dir, 'graph_labels.npy'))
train_idx = np.load(osp.join(data_dir, 'train_idx.npy'))
val_idx = np.load(osp.join(data_dir, 'val_idx.npy'))
test_idx = np.load(osp.join(data_dir, 'test_idx.npy'))
node_graph_id = np.load(osp.join(data_dir, 'node_graph_id.npy'))

print(f"✓ Loaded graph labels: {graph_labels.shape}")
print(f"✓ Train/Val/Test splits: {len(train_idx)}/{len(val_idx)}/{len(test_idx)}")

Loading gossipcop dataset with combined features: spacy, bert, content, profile
------------------------------------------------------------
✓ Loaded graph labels: (5464,)
✓ Train/Val/Test splits: 1092/546/3826


## 4. Load and Combine Features

In [4]:
# Load and combine all feature types
all_features = []
for feature in feature_types:
    feature_file = osp.join(data_dir, f'new_{feature}_feature.npz')
    feature_data = np.load(feature_file)
    sparse_feat = csr_matrix((feature_data['data'], feature_data['indices'], feature_data['indptr']), 
                              shape=tuple(feature_data['shape']))
    all_features.append(sparse_feat.toarray())
    print(f"✓ Loaded {feature} features: {sparse_feat.shape}")

# Concatenate all features
combined_features_array = np.concatenate(all_features, axis=1)
node_features = torch.FloatTensor(combined_features_array)
print(f"✓ Combined feature dimension: {node_features.shape}")
print()

✓ Loaded spacy features: (314262, 300)
✓ Loaded bert features: (314262, 768)
✓ Loaded content features: (314262, 310)
✓ Loaded profile features: (314262, 10)
✓ Combined feature dimension: torch.Size([314262, 1388])



## 5. Load Edge List

In [5]:
edges_list = []
adj_file = osp.join(data_dir, 'A.txt')
with open(adj_file, 'r') as f:
    for line in f:
        parts = line.strip().replace(',', ' ').split()
        if len(parts) >= 2:
            src, dst = int(parts[0]), int(parts[1])
            edges_list.append([src, dst])

edges_array = np.array(edges_list).T if edges_list else np.empty((2, 0), dtype=np.int64)
print(f"✓ Loaded edges: {edges_array.shape}")
print()

✓ Loaded edges: (2, 308798)



## 6. Create Graph Objects

In [6]:
def create_graphs_for_split(indices):
    """Create a list of Data objects for graphs in the given indices."""
    graphs = []
    
    for graph_id in indices:
        node_mask = (node_graph_id == graph_id)
        local_node_ids = np.where(node_mask)[0]
        
        if len(local_node_ids) == 0:
            continue
        
        node_id_mapping = {global_id: local_id for local_id, global_id in enumerate(local_node_ids)}
        
        graph_edges = []
        for src, dst in edges_array.T:
            if src in node_id_mapping and dst in node_id_mapping:
                local_src = node_id_mapping[src]
                local_dst = node_id_mapping[dst]
                graph_edges.append([local_src, local_dst])
        
        if len(graph_edges) > 0:
            edge_index = torch.LongTensor(np.array(graph_edges).T)
        else:
            edge_index = torch.LongTensor(2, 0)
        
        x = node_features[local_node_ids]
        y = torch.LongTensor([graph_labels[graph_id]])
        
        graph_data = Data(x=x, edge_index=edge_index, y=y)
        graphs.append(graph_data)
    
    return graphs

print("Creating train, val, and test datasets...")
train_dataset = create_graphs_for_split(train_idx)
val_dataset = create_graphs_for_split(val_idx)
test_dataset = create_graphs_for_split(test_idx)

print(f"✓ Train graphs: {len(train_dataset)}")
print(f"✓ Val graphs: {len(val_dataset)}")
print(f"✓ Test graphs: {len(test_dataset)}")
print()

Creating train, val, and test datasets...
✓ Train graphs: 1092
✓ Val graphs: 546
✓ Test graphs: 3826



## 7. Define SGConv Model

In [7]:
class SGConvGraphClassifier(nn.Module):
    """SGConv model for graph classification."""
    
    def __init__(self, input_dim, hidden_dim, num_classes, k=3, dropout=0.5):
        super(SGConvGraphClassifier, self).__init__()
        self.k = k
        self.conv = SGConv(input_dim, hidden_dim, K=k, cached=False)
        self.fc1 = nn.Linear(hidden_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(dropout)
        self.bn = nn.BatchNorm1d(hidden_dim)
        
    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        
        # SGConv layer
        x = self.conv(x, edge_index)
        x = self.bn(x)
        x = F.relu(x)
        
        # Global mean pooling
        x = global_mean_pool(x, batch)
        
        # Fully connected layers
        x = self.dropout(x)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x

print("✓ Model defined")

✓ Model defined


## 8. Hyperparameter Tuning

In [8]:
# Fixed Hyperparameters
k = 3
hidden_dim = 64
learning_rate = 0.01
batch_size = 32
num_epochs = 150
weight_decay = 1e-5
dropout = 0.5

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print()
print("Fixed Hyperparameters:")
print(f"  k: {k}")
print(f"  hidden_dim: {hidden_dim}")
print(f"  batch_size: {batch_size}")
print(f"  learning_rate: {learning_rate}")
print(f"  num_epochs: {num_epochs}")
print()

Device: cuda

Fixed Hyperparameters:
  k: 3
  hidden_dim: 64
  batch_size: 32
  learning_rate: 0.01
  num_epochs: 150



In [9]:
# Setup model training with fixed hyperparameters
print(f"\nTraining SGConv with fixed hyperparameters...")
print("-" * 60)

input_dim = train_dataset[0].x.shape[1]
num_classes = 2

model = SGConvGraphClassifier(input_dim, hidden_dim, num_classes, k=k, dropout=dropout)
model = model.to(device)

optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10)
criterion = nn.CrossEntropyLoss()

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Model total parameters: {sum(p.numel() for p in model.parameters()):,}")
print()

# Training loop
best_val_acc = 0
patience = 50
patience_counter = 0
train_losses = []
val_accs = []
test_accs = []

for epoch in range(1, num_epochs + 1):
    # Train
    model.train()
    train_loss = 0
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data)
        loss = criterion(out, data.y.squeeze())
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    train_losses.append(train_loss)
    
    # Validation
    model.eval()
    with torch.no_grad():
        val_preds = []
        val_labels = []
        for data in val_loader:
            data = data.to(device)
            out = model(data)
            pred = out.argmax(dim=1)
            val_preds.extend(pred.cpu().numpy())
            val_labels.extend(data.y.squeeze().cpu().numpy())
        
        val_acc = accuracy_score(val_labels, val_preds)
        val_accs.append(val_acc)
        
        # Test evaluation
        test_preds = []
        test_labels = []
        for data in test_loader:
            data = data.to(device)
            out = model(data)
            pred = out.argmax(dim=1)
            test_preds.extend(pred.cpu().numpy())
            test_labels.extend(data.y.squeeze().cpu().numpy())
        
        test_acc = accuracy_score(test_labels, test_preds)
        test_accs.append(test_acc)
    
    # Learning rate scheduling
    scheduler.step(val_acc)
    
    # Early stopping
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), 'best_sgconv_model.pt')
    else:
        patience_counter += 1
    
    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d} | Train Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f} | Test Acc: {test_acc:.4f}")
    
    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch}")
        break

print()


Training SGConv with fixed hyperparameters...
------------------------------------------------------------
Model total parameters: 93,314

Epoch   1 | Train Loss: 0.5028 | Val Acc: 0.8590 | Test Acc: 0.8361
Epoch  10 | Train Loss: 0.2354 | Val Acc: 0.9322 | Test Acc: 0.9234
Epoch  20 | Train Loss: 0.1041 | Val Acc: 0.9249 | Test Acc: 0.9190
Epoch  30 | Train Loss: 0.0421 | Val Acc: 0.9341 | Test Acc: 0.9454
Epoch  40 | Train Loss: 0.0236 | Val Acc: 0.9451 | Test Acc: 0.9548
Epoch  50 | Train Loss: 0.0311 | Val Acc: 0.9432 | Test Acc: 0.9511
Epoch  60 | Train Loss: 0.0213 | Val Acc: 0.9505 | Test Acc: 0.9501
Epoch  70 | Train Loss: 0.0079 | Val Acc: 0.9432 | Test Acc: 0.9475
Epoch  80 | Train Loss: 0.0091 | Val Acc: 0.9469 | Test Acc: 0.9501
Epoch  90 | Train Loss: 0.1795 | Val Acc: 0.9469 | Test Acc: 0.9490
Epoch 100 | Train Loss: 0.0684 | Val Acc: 0.9505 | Test Acc: 0.9506
Epoch 110 | Train Loss: 0.0019 | Val Acc: 0.9487 | Test Acc: 0.9511
Epoch 120 | Train Loss: 0.0117 | Val Acc: 0.

## 9. Display Results

In [10]:
print("="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"Best validation accuracy: {best_val_acc:.4f}")
print(f"Best test accuracy achieved: {max(test_accs):.4f}")
print(f"Average test accuracy (last 10 epochs): {np.mean(test_accs[-10:]) if len(test_accs) >= 10 else np.mean(test_accs):.4f}")
print(f"Total epochs trained: {len(train_losses)}")

TRAINING SUMMARY
Best validation accuracy: 0.9524
Best test accuracy achieved: 0.9548
Average test accuracy (last 10 epochs): 0.9499
Total epochs trained: 123


## 10. Train Final Model with Best Parameters

In [11]:
model.load_state_dict(torch.load('best_sgconv_model.pt'))

<All keys matched successfully>

## 11. Final Evaluation

In [12]:
model.eval()

print("\nFinal Evaluation on Test Set")
print("="*60)

with torch.no_grad():
    test_preds = []
    test_labels = []
    for data in test_loader:
        data = data.to(device)
        out = model(data)
        pred = out.argmax(dim=1)
        test_preds.extend(pred.cpu().numpy())
        test_labels.extend(data.y.squeeze().cpu().numpy())
    
    test_preds = np.array(test_preds)
    test_labels = np.array(test_labels)
    
    # Metrics
    accuracy = accuracy_score(test_labels, test_preds)
    precision = precision_score(test_labels, test_preds)
    recall = recall_score(test_labels, test_preds)
    f1 = f1_score(test_labels, test_preds)
    cm = confusion_matrix(test_labels, test_preds)
    
    print(f"\nTest Accuracy:  {accuracy:.4f}")
    print(f"Test Precision: {precision:.4f}")
    print(f"Test Recall:    {recall:.4f}")
    print(f"Test F1 Score:  {f1:.4f}")
    print(f"\nConfusion Matrix:")
    print(cm)
    print("✓ SGConv training complete!")


Final Evaluation on Test Set

Test Accuracy:  0.9509
Test Precision: 0.9435
Test Recall:    0.9593
Test F1 Score:  0.9513

Confusion Matrix:
[[1800  110]
 [  78 1838]]
✓ SGConv training complete!
